In [1]:
import random
import re
import warnings
import urllib.request

import numpy as np
import pandas as pd
import nltk

import pymorphy3
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

from gensim.models import Word2Vec, KeyedVectors

from corus import load_lenta
from navec import Navec

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [2]:
# Используем urllib для скачивания данных (если файл отсутствует)
data_url = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
local_filename = "lenta-ru-news.csv.gz"

try:
    with open(local_filename, "rb"):
        pass
except FileNotFoundError:
    print("Downloading dataset...")
    urllib.request.urlretrieve(data_url, local_filename)
    print("Download completed.")

Download completed.


In [3]:
# Загружаем данные с помощью Corus
raw_records = load_lenta(local_filename)
df = pd.DataFrame(raw_records)
df.columns = ['url', 'title', 'text', 'topic', 'tags', 'date']
df = df[['title', 'text', 'topic']]
df = df.sample(n=100000, random_state=SEED).reset_index(drop=True)
df.head()

,title,text,topic
0,EgyptAir объявила о подорожании билетов,Египетский перевозчик EgyptAir сообщил о возмо...,Путешествия
1,Глава Красногорского района Подмосковья ушел в...,Глава Красногорского района Московской области...,Россия
2,Милонов предложил запретить россиянам сидеть в...,Депутат Виталий Милонов внес в Госдуму законоп...,Россия
3,Женщинам в детородном возрасте разрешили посещ...,Верховный суд Индии разрешил женщинам в фертил...,Мир
4,Россиянам пообещали дешевый хлеб,Россиянам не стоит бояться роста цен на хлеб —...,Экономика


In [4]:
# Фильтрация редких классов: оставляем только топики с не менее чем 1 примерами
topic_freq = df['topic'].value_counts()
popular_topics = topic_freq[topic_freq > 1].index
df = df[df['topic'].isin(popular_topics)].reset_index(drop=True)
df['topic'].value_counts()

topic
Россия               21871
Мир                  18494
Экономика            10737
Спорт                 8632
Культура              7337
Наука и техника       7129
Бывший СССР           7100
Интернет и СМИ        6181
Из жизни              3718
Дом                   2891
Силовые структуры     2661
Ценности              1079
Бизнес                 967
Путешествия            855
69-я параллель         178
Крым                    82
Культпросвет            45
                        23
Легпром                 10
Библиотека               8
Name: count, dtype: int64

In [5]:
# Загрузка инструментов для обработки текста
nltk.download('stopwords')
russian_stop = set(stopwords.words("russian"))
morph = pymorphy3.MorphAnalyzer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ivann\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [6]:
def clean_text(raw_text: str) -> str:
    txt = raw_text.lower()
    txt = re.sub(r"<.*?>", " ", txt)
    txt = re.sub(r"[^а-яё\s]", " ", txt)
    tokens = [w for w in txt.split() if w not in russian_stop]
    tokens = [morph.parse(word)[0].normal_form for word in tokens]
    return " ".join(tokens)

Базовая предобработка, включающая очистку (удаление HTML-тегов, перевод в нижний регистр, фильтрация символов и удаление стоп-слов) и лемматизация. Базовая обработка для удаления шума из данных и снижения размерности признакового пространства. Сохраняю только русские слова для снижения размера признаков (наверняка слова английские встречаются слишком редко + далее используется pymorphy3.MorphAnalyzer() только для русских слов для тегов для эмбеддингов rusvectores).

In [7]:
df["full_text"] = (df["title"] + " " + df["text"]).apply(clean_text)
df["full_text"].head()

0    объявить подорожание билет египетский перевозч...
1    глава красногорский район подмосковье уйти отс...
2    милон предложить запретить россиянин сидеть со...
3    женщина детородный возраст разрешить посещать ...
4    россиянин пообещать дешёвый хлеб россиянин сто...
Name: full_text, dtype: object

In [8]:
# Разбиваем данные на train/validation/test (60/20/20) с сохранением пропорций классов
X = df["full_text"]
y = df["topic"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, train_size=0.6, stratify=y, random_state=SEED)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

## Train Word2Vec embeddings

Параметры модели:
- vector_size=300: увеличенная размерность для более точного семантического представления и сравнения с другими моделями, так как модели размером 300.
- window=5: стандартное окно, захватывающее 5 соседних слов для контекстуального обучения.
- min_count=10: игнорирование редких слов для уменьшения шума.
- sg=1: использование архитектуры skip-gram, лучше отражающей семантику редких слов.
- epochs=15: достаточное число эпох для сходимости без переобучения.
- seed=SEED: фиксация случайного состояния для воспроизводимости.

In [9]:
w2v_model = Word2Vec(
    [text.split() for text in X_train],
    vector_size=300,
    window=5,
    min_count=10,
    sg=1,
    epochs=15,
    seed=SEED
)

In [10]:
words_group_new1 = ["яблоко", "банан", "вишня", "стул"]
mismatch_new1 = w2v_model.wv.doesnt_match(words_group_new1)
print(f"Doesn't match from {words_group_new1}: {mismatch_new1}")

words_group_new2 = ["учитель", "директор", "студент", "телефон"]
mismatch_new2 = w2v_model.wv.doesnt_match(words_group_new2)
print(f"Doesn't match from {words_group_new2}: {mismatch_new2}")

similar_words_new1 = w2v_model.wv.most_similar("спорт", topn=5)
print("\nMost similar words for 'спорт':")
for word, score in similar_words_new1:
    print(f"{word}: {score:.3f}")

similar_words_new2 = w2v_model.wv.most_similar("музыка", topn=5)
print("\nMost similar words for 'музыка':")
for word, score in similar_words_new2:
    print(f"{word}: {score:.3f}")

Doesn't match from ['яблоко', 'банан', 'вишня', 'стул']: стул
Doesn't match from ['учитель', 'директор', 'студент', 'телефон']: директор

Most similar words for 'спорт':
конькобежный: 0.529
мутко: 0.520
флгра: 0.507
ффккра: 0.504
шляхтин: 0.494

Most similar words for 'музыка':
музыкальный: 0.593
песня: 0.521
композитор: 0.520
аранжировка: 0.509
блюзовый: 0.499


Визуально эмбеддинги довольно хорошие, но есть проблемы - Doesn't match from ['учитель', 'директор', 'студент', 'телефон']: директор

## Предобученные эмбеддинги из navec и rusvectores

In [11]:
# Navec embeddings
navec_url = "https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar"
navec_local_filename = "navec_hudlit.tar"

try:
    with open(navec_local_filename, "rb"):
        pass
except FileNotFoundError:
    print("Downloading Navec embeddings...")
    urllib.request.urlretrieve(navec_url, navec_local_filename)
    print("Navec download completed.")

# RusVectores embeddings
rusvectores_url = "https://rusvectores.org/static/models/rusvectores4/ruwikiruscorpora/ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz"
rusvectores_local_filename = "ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz"

try:
    with open(rusvectores_local_filename, "rb"):
        pass
except FileNotFoundError:
    print("Downloading RusVectores embeddings...")
    urllib.request.urlretrieve(rusvectores_url, rusvectores_local_filename)
    print("RusVectores download completed.")


Navec download completed.
RusVectores download completed.


In [12]:
navec = Navec.load("navec_hudlit.tar")
rusvectores = KeyedVectors.load_word2vec_format('ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz')

In [13]:
def average_embedding(text, embedding, vec_size):
    tokens = text.split()
    vectors = [embedding[token] for token in tokens if token in embedding]
    return np.mean(vectors, axis=0) if vectors else np.zeros(vec_size)

In [14]:
def average_embedding_rusvectores(text, embedding, vec_size):
    tokens = text.split()
    vectors = []
    for token in tokens:
        pos_tag = morph.parse(token)[0].tag.POS 
        token_tagged = f"{token}_{pos_tag}" if pos_tag else token
        if token_tagged in embedding:
            vectors.append(embedding[token_tagged])
        elif token in embedding:
            vectors.append(embedding[token])
    return np.mean(vectors, axis=0) if vectors else np.zeros(vec_size)

In [15]:
X_train_w2v = np.vstack(X_train.apply(lambda x: average_embedding(x, w2v_model.wv, 300)))
X_val_w2v   = np.vstack(X_valid.apply(lambda x: average_embedding(x, w2v_model.wv, 300)))
X_test_w2v  = np.vstack(X_test.apply(lambda x: average_embedding(x, w2v_model.wv, 300)))

X_train_navec = np.vstack(X_train.apply(lambda x: average_embedding(x, navec, 300)))
X_val_navec   = np.vstack(X_valid.apply(lambda x: average_embedding(x, navec, 300)))
X_test_navec  = np.vstack(X_test.apply(lambda x: average_embedding(x, navec, 300)))

X_train_rusvectores = np.vstack(X_train.apply(lambda x: average_embedding_rusvectores(x, rusvectores, 300)))
X_val_rusvectores   = np.vstack(X_valid.apply(lambda x: average_embedding_rusvectores(x, rusvectores, 300)))
X_test_rusvectores  = np.vstack(X_test.apply(lambda x: average_embedding_rusvectores(x, rusvectores, 300)))

## LogisticRegression с тремя вариантами векторизации текстов

In [16]:
# For plain word2vec model (without TF-IDF)
lr_w2v = LogisticRegression(random_state=SEED, max_iter=1000)
lr_w2v.fit(X_train_w2v, y_train)
y_val_pred_w2v = lr_w2v.predict(X_val_w2v)
val_accuracy_w2v = accuracy_score(y_valid, y_val_pred_w2v)

print(f"\nValidation accuracy (word2vec): {val_accuracy_w2v:.4f}")
print("\nClassification report for word2vec:")
print(classification_report(y_valid, y_val_pred_w2v))


Validation accuracy (word2vec): 0.7909

Classification report for word2vec:
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       1.00      0.17      0.29        36
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.67      0.11      0.19       193
      Бывший СССР       0.80      0.77      0.79      1420
              Дом       0.84      0.73      0.78       578
         Из жизни       0.63      0.54      0.58       743
   Интернет и СМИ       0.75      0.67      0.71      1236
             Крым       1.00      0.06      0.12        16
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.86      0.86      0.86      1468
          Легпром       0.00      0.00      0.00         2
              Мир       0.78      0.84      0.81      3699
  Наука и техника       0.81      0.80      0.80      1426
      Путешествия       0.72      0.5

In [17]:
# For navec pre-trained embeddings
lr_navec = LogisticRegression(random_state=SEED, max_iter=1000)
lr_navec.fit(X_train_navec, y_train)
y_val_pred_navec = lr_navec.predict(X_val_navec)
val_accuracy_navec = accuracy_score(y_valid, y_val_pred_navec)

print(f"\nValidation accuracy (navec): {val_accuracy_navec:.4f}")
print("\nClassification report for navec:")
print(classification_report(y_valid, y_val_pred_navec))


Validation accuracy (navec): 0.7632

Classification report for navec:
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       0.71      0.14      0.23        36
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.53      0.09      0.15       193
      Бывший СССР       0.76      0.69      0.73      1420
              Дом       0.78      0.70      0.74       578
         Из жизни       0.60      0.52      0.55       743
   Интернет и СМИ       0.70      0.64      0.67      1236
             Крым       1.00      0.06      0.12        16
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.84      0.86      0.85      1468
          Легпром       0.00      0.00      0.00         2
              Мир       0.77      0.82      0.79      3699
  Наука и техника       0.78      0.77      0.77      1426
      Путешествия       0.63      0.47     

In [18]:
# For RusVectores pre-trained embeddings (using POS tags)
lr_rusvectores = LogisticRegression(random_state=SEED, max_iter=1000)
lr_rusvectores.fit(X_train_rusvectores, y_train)
y_val_pred_rusvectores = lr_rusvectores.predict(X_val_rusvectores)
val_accuracy_rusvectores = accuracy_score(y_valid, y_val_pred_rusvectores)

print(f"\nValidation accuracy (rusvectores): {val_accuracy_rusvectores:.4f}")
print("\nClassification report for rusvectores:")
print(classification_report(y_valid, y_val_pred_rusvectores))



Validation accuracy (rusvectores): 0.7389

Classification report for rusvectores:
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       0.00      0.00      0.00        36
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.00      0.00      0.00       193
      Бывший СССР       0.76      0.55      0.64      1420
              Дом       0.79      0.57      0.66       578
         Из жизни       0.58      0.42      0.49       743
   Интернет и СМИ       0.71      0.60      0.65      1236
             Крым       0.00      0.00      0.00        16
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.81      0.84      0.82      1468
          Легпром       0.00      0.00      0.00         2
              Мир       0.73      0.83      0.78      3699
  Наука и техника       0.75      0.76      0.76      1426
      Путешествия       0.74   

Лучше показали себя эмбеддинги, обученные на модели Word2Vec

## TF IDF взвешивание

In [19]:
tfidf_vect = TfidfVectorizer()
tfidf_vect.fit(X_train)

TfidfVectorizer()

In [20]:
def get_tfidf_weighted_avg_vector(text, embedding_model, vector_size, tfidf_vectorizer):
    tokens = text.split()
    weighted_vec = np.zeros(vector_size)
    weight_sum = 0.0
    for token in tokens:
        if token in embedding_model and token in tfidf_vectorizer.vocabulary_:
            idx = tfidf_vectorizer.vocabulary_[token]
            weight = tfidf_vectorizer.idf_[idx]
            weighted_vec += embedding_model[token] * weight
            weight_sum += weight
    if weight_sum != 0:
        weighted_vec /= weight_sum
    return weighted_vec

In [21]:
X_train_tfidf = np.vstack(X_train.apply(lambda x: get_tfidf_weighted_avg_vector(x, w2v_model.wv, w2v_model.vector_size, tfidf_vect)))
X_val_tfidf   = np.vstack(X_valid.apply(lambda x: get_tfidf_weighted_avg_vector(x, w2v_model.wv, w2v_model.vector_size, tfidf_vect)))
X_test_tfidf  = np.vstack(X_test.apply(lambda x: get_tfidf_weighted_avg_vector(x, w2v_model.wv, w2v_model.vector_size, tfidf_vect)))

In [22]:
lr_tfidf_v2w = LogisticRegression(random_state=SEED, max_iter=1000)
lr_tfidf_v2w.fit(X_train_tfidf, y_train)
y_val_pred_tfidf_v2w = lr_tfidf_v2w.predict(X_val_tfidf)
val_accuracy_tfidf_v2w = accuracy_score(y_valid, y_val_pred_tfidf_v2w)

print(f"Validation accuracy (tf-idf w2v_model): {val_accuracy_tfidf_v2w:.4f}")
print("\nClassification report for tf-idf w2v_model:")
print(classification_report(y_valid, y_val_pred_tfidf_v2w))

Validation accuracy (tf-idf w2v_model): 0.7892

Classification report for tf-idf w2v_model:
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       1.00      0.28      0.43        36
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.62      0.12      0.20       193
      Бывший СССР       0.80      0.76      0.78      1420
              Дом       0.84      0.74      0.78       578
         Из жизни       0.62      0.54      0.58       743
   Интернет и СМИ       0.73      0.67      0.70      1236
             Крым       1.00      0.06      0.12        16
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.85      0.87      0.86      1468
          Легпром       0.00      0.00      0.00         2
              Мир       0.78      0.84      0.81      3699
  Наука и техника       0.80      0.80      0.80      1426
      Путешествия     

Эмбеддинги с tf-idf взвешиванием показали себя хуже чем с обычным взвешиванием

## Метрики на тесте

In [23]:
# Для модели word2vec с TF-IDF
y_test_pred_tfidf_v2w = lr_tfidf_v2w.predict(X_test_tfidf)
test_accuracy_tfidf_v2w = accuracy_score(y_test, y_test_pred_tfidf_v2w)
print("Отчет о классификации для word2vec с TF-IDF:")
print(classification_report(y_test, y_test_pred_tfidf_v2w))

Отчет о классификации для word2vec с TF-IDF:
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         4
   69-я параллель       0.88      0.20      0.33        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.47      0.14      0.22       194
      Бывший СССР       0.79      0.75      0.77      1420
              Дом       0.84      0.78      0.81       578
         Из жизни       0.64      0.57      0.60       744
   Интернет и СМИ       0.72      0.66      0.69      1236
             Крым       1.00      0.06      0.11        17
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.86      0.85      0.85      1467
          Легпром       0.00      0.00      0.00         2
              Мир       0.77      0.82      0.80      3699
  Наука и техника       0.79      0.81      0.80      1426
      Путешествия       0.73      0.59      0.65       171
          

In [24]:
# Для модели word2vec (без TF-IDF)
y_test_pred_w2v = lr_w2v.predict(X_test_w2v)
test_accuracy_w2v = accuracy_score(y_test, y_test_pred_w2v)
print("Отчет о классификации для word2vec:")
print(classification_report(y_test, y_test_pred_w2v))


Отчет о классификации для word2vec:
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         4
   69-я параллель       0.75      0.09      0.15        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.38      0.09      0.15       194
      Бывший СССР       0.80      0.76      0.78      1420
              Дом       0.84      0.80      0.82       578
         Из жизни       0.65      0.57      0.60       744
   Интернет и СМИ       0.73      0.66      0.69      1236
             Крым       0.00      0.00      0.00        17
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.86      0.85      0.86      1467
          Легпром       0.00      0.00      0.00         2
              Мир       0.78      0.82      0.80      3699
  Наука и техника       0.78      0.81      0.80      1426
      Путешествия       0.74      0.61      0.67       171
           Россия  

In [25]:
# Для модели Navec (предобученные эмбеддинги)
y_test_pred_navec = lr_navec.predict(X_test_navec)
test_accuracy_navec = accuracy_score(y_test, y_test_pred_navec)
print("Отчет о классификации для navec:")
print(classification_report(y_test, y_test_pred_navec))

Отчет о классификации для navec:
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         4
   69-я параллель       0.60      0.09      0.15        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.41      0.08      0.14       194
      Бывший СССР       0.75      0.68      0.71      1420
              Дом       0.79      0.78      0.78       578
         Из жизни       0.65      0.53      0.58       744
   Интернет и СМИ       0.70      0.64      0.67      1236
             Крым       0.00      0.00      0.00        17
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.84      0.85      0.85      1467
          Легпром       0.00      0.00      0.00         2
              Мир       0.76      0.81      0.78      3699
  Наука и техника       0.76      0.78      0.77      1426
      Путешествия       0.68      0.49      0.57       171
           Россия     

In [26]:
# Для модели RusVectores (предобученные эмбеддинги с POS-тегированием)
y_test_pred_rusvectores = lr_rusvectores.predict(X_test_rusvectores)
test_accuracy_rusvectores = accuracy_score(y_test, y_test_pred_rusvectores)
print("Отчет о классификации для rusvectores:")
print(classification_report(y_test, y_test_pred_rusvectores))


Отчет о классификации для rusvectores:
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         4
   69-я параллель       0.00      0.00      0.00        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.00      0.00      0.00       194
      Бывший СССР       0.75      0.53      0.62      1420
              Дом       0.81      0.65      0.72       578
         Из жизни       0.62      0.43      0.51       744
   Интернет и СМИ       0.69      0.60      0.64      1236
             Крым       0.00      0.00      0.00        17
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.82      0.84      0.83      1467
          Легпром       0.00      0.00      0.00         2
              Мир       0.72      0.81      0.76      3699
  Наука и техника       0.73      0.78      0.76      1426
      Путешествия       0.67      0.09      0.16       171
           Росси

In [27]:
print(f"Тестовая точность (word2vec с TF-IDF): {test_accuracy_tfidf_v2w:.4f}")
print(f"Тестовая точность (word2vec): {test_accuracy_w2v:.4f}")
print(f"Тестовая точность (navec): {test_accuracy_navec:.4f}")
print(f"Тестовая точность (rusvectores): {test_accuracy_rusvectores:.4f}")

Тестовая точность (word2vec с TF-IDF): 0.7839
Тестовая точность (word2vec): 0.7873
Тестовая точность (navec): 0.7610
Тестовая точность (rusvectores): 0.7325


На тестовой выборке модель word2vec с обычным усреднением показала лучшие метрики, что объясняется тем, что, несмотря на разбиение данных, обучение и тест проводились на одном и том же корпусе новостей. Для более объективной оценки следовало бы использовать другой новостной корпус. Также усреднение с TF-IDF не принесло прибавки в качестве.